In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pymc as pm
import arviz as az
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
from numba import njit
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.cluster import AgglomerativeClustering
import itertools
import os
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = gpd.read_file('../data/updated_data.geojson')

In [ ]:
y = df["fem_empl_rate"].to_numpy()
y = y / 100
y_star = np.log(y / (1 - y))

selected_covariates = ['fertility_rate', 'young_neet', 'coverage', 'per_capita_public_expenditure', 'other_percapita_user_contrib']

X = df[selected_covariates].to_numpy()
X_scaler = StandardScaler()
X_star = X_scaler.fit_transform(X)

median_y = np.median(y_star)
std_y = np.std(y_star)

N = 107 
P = len(selected_covariates) 
K = 20

In [ ]:
print("Median of y_star:", median_y)
print("Std of y_star:", std_y)

In [ ]:
df = df.reset_index(drop=True)

n = len(df)
node1 = []
node2 = []
num_neighbors = np.zeros(n, dtype=int)

sindex = df.sindex  # spatial index

for i, geom in enumerate(df.geometry):
    # candidates using bounding boxes
    possible_matches = list(sindex.intersection(geom.bounds))

    for j in possible_matches:
        if j <= i:
            continue
        if geom.touches(df.geometry[j]):
            node1.append(i + 1)   # Stan is 1-based
            node2.append(j + 1)

            num_neighbors[i] += 1
            num_neighbors[j] += 1

node1 = np.asarray(node1, dtype=int)
node2 = np.asarray(node2, dtype=int)

W = np.zeros((n, n), dtype=int)

for i, j in zip(node1 - 1, node2 - 1):
    W[i, j] = 1
    W[j, i] = 1

## Label Assignment

In [ ]:
mix_model = """
functions {
  real car_leroux_lpdf(
    vector phi,
    real tau2,
    real rho,
    matrix W,
    vector D,
    vector lambda,
    int N
  ) {
    vector[N] Dphi;
    vector[N] Wphi;
    vector[N] det_terms;
    real quad;

    Dphi = D .* phi;        // D * phi
    Wphi = W * phi;         // W * phi

    for (i in 1 : N) {
      det_terms[i] = log(rho * lambda[i] + (1 - rho)); // eigenvalues of D-W
    }

    quad = rho * (dot_product(phi, Dphi) - dot_product(phi, Wphi)) + (1 - rho) * dot_product(phi, phi);

    return 0.5 * (
      N * log(tau2)
      + sum(det_terms)  
      - tau2 * quad
    );
  }
}

data {
  int<lower=1> N;          
  int<lower=1> P;          
  int<lower=1> K;  
  real std_y;
  real median_y;        
  vector[N] y_star;             
  matrix[N, P] X_star;           
  matrix<lower=0, upper=1>[N, N] W;  
}

transformed data {
  vector[N] D;            // diagonal entries of D
  vector[N] lambda;       // eigenvalues of D-W

  for (i in 1:N)
    D[i] = sum(W[i]);

  lambda = eigenvalues_sym(diag_matrix(D) - W);      // eigenvalues
}

parameters {
  simplex[K] pi;           
  real alpha;  
  vector[P] b0;
  vector<lower = 1e-6, upper=100>[P] lambda_b;      
  matrix[P, K] beta; 
  vector<lower=1e-6>[K] tau;  
  real<lower=1e-6> tau2;                  // CAR precision
  vector[N] phi_raw;                        // Spatial random effects
  real<lower=0, upper=0.95> rho;      // Leroux dependence parameter
}

transformed parameters {
  vector[N] phi;
  phi = phi_raw - mean(phi_raw);

  vector<lower=0>[K] sigma = inv_sqrt(tau); 

  vector<lower=0>[P] sd_beta = std_y * sqrt(lambda_b); 
}


model {
  // Priors

  pi ~ dirichlet(rep_vector(0.5, K)); 

  for (j in 1:P) {
    b0[j] ~ normal(0, 5);
    lambda_b[j] ~ gamma(0.5, 0.5);
  }

  for(k in 1:K) {
    tau[k] ~ gamma(2, 1);
  }

  alpha ~ normal(0, 5);

  for (j in 1:P) {
    for (k in 1:K) {
      beta[j, k] ~ normal(b0[j], sd_beta[j]);
    }
  }


  // CAR prior
  tau2  ~ gamma(2, 1);
  rho   ~ beta(1, 1);
  phi_raw ~ car_leroux(tau2, rho, W, D, lambda, N);

  // Likelihood
  for (i in 1:N) {
    vector[K] lps;
    for (k in 1:K) {
      real mu_ik;
      mu_ik = alpha + dot_product(row(X_star, i), beta[, k]) + phi[i];
      lps[k] = log(pi[k]) + normal_lpdf(y_star[i] | mu_ik, sigma[k]);
    }
    target += log_sum_exp(lps);
  }
}

generated quantities {
  array[N] int z;

  for (i in 1:N) {
    vector[K] lps;
    vector[K] r_i;

    for (k in 1:K) {
      real mu_ik = alpha + dot_product(row(X_star, i), beta[, k]) + phi[i];
      lps[k] = log(pi[k]) + normal_lpdf(y_star[i] | mu_ik, sigma[k]);
    }

    r_i = softmax(lps);
    z[i] = categorical_rng(r_i);
    
  }
}

"""

In [ ]:
with open("normal_mixture.stan", "w") as f:
    f.write(mix_model)

In [ ]:
data = {
    "N": int(N),
    "P": int(P),
    "K": int(K),
    "std_y": float(std_y),
    "median_y": float(median_y),
    "y_star": y_star.astype(float),
    "X_star": X_star.astype(float),
    "W": W,
}

In [ ]:
mix_stan = CmdStanModel(stan_file="normal_mixture.stan")

In [ ]:
fit = mix_stan.sample(
    data=data,
    chains=4,
    iter_warmup=3000,
    iter_sampling=2000,
    adapt_delta=0.99,
    max_treedepth=18,
)

In [ ]:
summary_beta_95 = az.summary(fit, var_names=["beta", "phi", "rho"])
print(summary_beta_95)
phi_mean = summary_beta_95.loc[summary_beta_95.index.str.startswith("phi"), "mean"].values
plt.hist(phi_mean, bins=30)

betas_mean = summary_beta_95.loc[summary_beta_95.index.str.startswith("beta"), "mean"].values

In [ ]:
betas_mean

In [ ]:
import geopandas as gpd

# gdf: GeoDataFrame with one row per observation/region
# must have a geometry column of polygons/multipolygons
gdf = gpd.read_file("../data/updated_data.geojson")
gdf["phi_mean"] = phi_mean

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

gdf.plot(column="phi_mean", categorical=False, legend=True, ax=ax1)
ax1.set_title("Spatial effects (phi mean)")
ax1.set_axis_off()

gdf.plot(column="fem_empl_rate", categorical=False, legend=True, ax=ax2)
ax2.set_title("Female employment rate")
ax2.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
pd.DataFrame(betas_mean, columns=['beta']).sort_values(by='beta').plot(kind='barh')
plt.title('Lasso-selected variables influencing female employment rate')
plt.xlabel('Coefficient (standardized scale)')
plt.show()

In [ ]:
az.rcParams["plot.max_subplots"] = 1000
az.plot_trace(fit, compact=False)
plt.tight_layout()

In [ ]:
fit.stan_variables().keys()

In [ ]:
z_draws = fit.stan_variable("z")   # shape: (n_draws, N)

In [ ]:
## For each draw:
# - compare couples of observations
# - if they are equal (same label) increment the counter
# - normalize

# At the end psm[i, j] is the posterior probability that observations i and j belong to the same cluster

# Shape is n_draws x n_obs, and the numbers are class labels

@njit
def build_psm(posterior):
    n_draws, n_obs = posterior.shape
    psm = np.zeros((n_obs, n_obs))
    for row in posterior:
        for r_idx, i in enumerate(row):
            for c_idx, j in enumerate(row):
                if i == j:
                    psm[r_idx, c_idx] += 1
                    
    psm /= n_draws
        
    return psm

In [ ]:
psm = build_psm(z_draws.astype(np.int64))

In [ ]:
def order_by_hclust(psm, method="average"):
    # convert similarity -> distance
    D = 1.0 - psm
    np.fill_diagonal(D, 0.0)
    D = np.clip(D, 0.0, 1.0)

    # linkage needs condensed distances
    condensed = squareform(D, checks=False)
    Z = linkage(condensed, method=method)
    order = leaves_list(Z)   # permutation of 0..N-1
    return order

order = order_by_hclust(psm)
psm_sorted = psm[np.ix_(order, order)]

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(psm_sorted, center=1)
plt.show()

In [ ]:
def binder_loss(psm, labels):
    # expected Binder loss from PSM (sum over i<j)
    labels = np.asarray(labels)
    A = (labels[:, None] == labels[None, :]).astype(float)
    iu = np.triu_indices(psm.shape[0], k=1)
    return float(np.sum(np.abs(A[iu] - psm[iu])))

def binder_optimal_partition(psm, k_min, k_max):
    psm = np.asarray(psm, dtype=float)
    psm = 0.5 * (psm + psm.T)
    np.fill_diagonal(psm, 1.0)

    dist = 1.0 - psm
    np.fill_diagonal(dist, 0.0)

    best_loss = np.inf
    best_labels = None

    for k in range(k_min, min(k_max, psm.shape[0]) + 1):
        model = AgglomerativeClustering(
            n_clusters=k,
            linkage="average",
            metric="precomputed",  
        )
        labels = model.fit_predict(dist)
        loss = binder_loss(psm, labels)
        if loss < best_loss:
            best_loss = loss
            best_labels = labels

    return best_labels, best_loss

In [ ]:
labels_opt, loss_opt = binder_optimal_partition(psm, k_min=1, k_max=20)

In [ ]:
loss_opt

In [ ]:
labels_opt

In [ ]:
labels = labels_opt + 1

In [ ]:
labels

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# gdf: GeoDataFrame with one row per observation/region
# must have a geometry column of polygons/multipolygons
gdf = gpd.read_file("../data/updated_data.geojson")
gdf["cluster_bayes"] = labels

C = 7
kmeans = KMeans(n_clusters=C, random_state=0, n_init="auto").fit(X_star)
gdf["cluster"] = kmeans.labels_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

gdf.plot(column="cluster_bayes", categorical=True, legend=True, ax=ax1)
ax1.set_title("Bayes clusters")
ax1.set_axis_off()

gdf.plot(column="cluster", categorical=True, legend=True, ax=ax2)
ax2.set_title("KMeans clusters")
ax2.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
with open("labels_car.txt", "w") as f:
    for x in labels:
        f.write(f"{x}\n")

In [ ]:
gdf["cluster"] = gdf["cluster"] + 1

In [ ]:
gdf["cluster"].value_counts()

## "Final" Model

Actually we still have to add in the model the spatial effect

In [ ]:
model = """
functions {
  real car_leroux_lpdf(
    vector phi,
    real tau2,
    real rho,
    matrix W,
    vector D,
    vector lambda,
    int N
  ) {
    vector[N] Dphi;
    vector[N] Wphi;
    vector[N] det_terms;
    real quad;

    Dphi = D .* phi;        // D * phi
    Wphi = W * phi;         // W * phi

    for (i in 1 : N) {
      det_terms[i] = log(rho * lambda[i] + (1 - rho)); // eigenvalues of D-W
    }

    quad = rho * (dot_product(phi, Dphi) - dot_product(phi, Wphi)) + (1 - rho) * dot_product(phi, phi);

    return 0.5 * (
      N * log(tau2)
      + sum(det_terms)  
      - tau2 * quad
    );
  }
}

data {
  int<lower=1> N;          
  int<lower=1> P;          
  int<lower=1> C;
  array[N] int<lower=1, upper=C> cluster;  
  real std_y;
  real median_y;        
  vector[N] y_star;             
  matrix[N, P] X_star;           
  matrix<lower=0, upper=1>[N, N] W;  
}

transformed data {
  vector[N] D;            // diagonal entries of D
  vector[N] lambda;       // eigenvalues of D-W

  for (i in 1:N)
    D[i] = sum(W[i]);

  lambda = eigenvalues_sym(diag_matrix(D) - W);      // eigenvalues
}

parameters {           
  real alpha;  
  vector[P] b0;
  vector<lower = 1e-6, upper=100>[P] lambda_b;      
  matrix[P, C] beta; 
  vector<lower=1e-6>[C] tau;  
  real<lower=1e-6> tau2;                  // CAR precision
  vector[N] phi_raw;                        // Spatial random effects
  real<lower=0, upper=0.95> rho;      // Leroux dependence parameter
}

transformed parameters {
  vector[N] phi;
  phi = phi_raw - mean(phi_raw);

  vector<lower=0>[C] sigma = inv_sqrt(tau); 

  vector<lower=0>[P] sd_beta = std_y * sqrt(lambda_b); 
}


model {
  // Priors
  for (j in 1:P) {
    b0[j] ~ normal(0, 5);
    lambda_b[j] ~ gamma(0.5, 0.5);
  }

  for(k in 1:C) {
    tau[k] ~ gamma(2, 1);
  }

  alpha ~ normal(0, 5);

  for (j in 1:P) {
    for (k in 1:C) {
      beta[j, k] ~ normal(b0[j], sd_beta[j]);
    }
  }


  // CAR prior
  tau2  ~ gamma(2, 1);
  rho   ~ beta(1, 1);
  phi_raw ~ car_leroux(tau2, rho, W, D, lambda, N);

  // Likelihood
  for (i in 1:N) {

    int c = cluster[i];
    real mu = alpha + dot_product(row(X_star, i), beta[, c]) + phi[i];
    target += normal_lpdf(y_star[i] | mu, sigma[c]);
  }
}

"""

In [ ]:
with open("labeled_model.stan", "w") as f:
    f.write(model)

In [ ]:
data_m = {
    "N": int(N),
    "P": int(P),
    "C": int(C),
    "cluster": gdf["cluster"].astype(int),
    "std_y": float(std_y),
    "median_y": float(median_y),
    "y_star": y_star.astype(float),
    "X_star": X_star.astype(float),
    "W": W,
}

In [ ]:
lab_model = CmdStanModel(stan_file="labeled_model.stan")

In [ ]:
fit_l = lab_model.sample(
    data=data_m,
    chains=4,
    iter_warmup=3000,
    iter_sampling=2000,
    adapt_delta=0.99,
    max_treedepth=18,
)

In [ ]:
az.rcParams["plot.max_subplots"] = 1000
az.plot_trace(fit_l, compact=False)
plt.tight_layout()